# Notebook 02 — AutoML Pipeline
**AI Data Intelligence Copilot**

This notebook demonstrates:
- Dataset preprocessing
- Multi-model AutoML training
- Model leaderboard
- Performance evaluation (classification + regression)
- ROC curves, confusion matrices, predicted vs actual

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from src.ingestion.schema_detector import detect_schema
from src.automl.preprocessor import preprocess
from src.automl.trainer import train_all
from src.automl.evaluator import (
    plot_confusion_matrix, plot_roc_curve,
    plot_predicted_vs_actual, plot_residuals, plot_leaderboard
)
from src.automl.model_selector import get_selection_rationale

print('Imports OK')

## Part A — Classification (Telco Churn)

In [ ]:
df = pd.read_csv('../data/sample_datasets/telco_churn.csv')
schema = detect_schema(df)
TARGET = 'Churn'

print(f'Problem type: {schema.problem_type}')
print(f'Features: {len(schema.feature_cols(TARGET))}')
df[TARGET].value_counts()

In [ ]:
# Preprocess
prep = preprocess(df, TARGET, schema)
print(f'Train set: {prep.X_train.shape}')
print(f'Test set : {prep.X_test.shape}')
print(f'Features : {prep.feature_names}')

In [ ]:
# Train all models
results = train_all(
    prep.X_train, prep.X_test,
    prep.y_train, prep.y_test,
    schema.problem_type
)

# Show leaderboard (without estimator objects)
lb_df = pd.DataFrame([
    {k: v for k, v in m.items() if k != 'estimator'}
    for m in results['leaderboard']
])
print(lb_df.to_string(index=False))

In [ ]:
# Selection rationale
print(get_selection_rationale(results['leaderboard'], schema.problem_type))

In [ ]:
# Leaderboard chart
plot_leaderboard(results['leaderboard'], schema.problem_type).show()

In [ ]:
# Best model evaluation
best = results['best_model']['estimator']
class_names = [str(c) for c in prep.label_encoder.classes_]

plot_confusion_matrix(best, prep.X_test, prep.y_test, class_names).show()
plot_roc_curve(best, prep.X_test, prep.y_test).show()

## Part B — Regression (Boston Housing)

In [ ]:
df_reg = pd.read_csv('../data/sample_datasets/boston_housing.csv')
schema_reg = detect_schema(df_reg)
schema_reg.problem_type = 'regression'
TARGET_REG = 'MEDV'

prep_reg = preprocess(df_reg, TARGET_REG, schema_reg)
results_reg = train_all(
    prep_reg.X_train, prep_reg.X_test,
    prep_reg.y_train, prep_reg.y_test,
    'regression'
)

lb_reg = pd.DataFrame([
    {k: v for k, v in m.items() if k != 'estimator'}
    for m in results_reg['leaderboard']
])
print(lb_reg.to_string(index=False))

In [ ]:
best_reg = results_reg['best_model']['estimator']
plot_predicted_vs_actual(best_reg, prep_reg.X_test, prep_reg.y_test).show()
plot_residuals(best_reg, prep_reg.X_test, prep_reg.y_test).show()